# Energy Consumption Forecasting 

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.seasonal import STL

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True


## 1. Load Data

In [ ]:
# On Kaggle, the input file typically lives under /kaggle/input/...
# Adjust the path below to match your Kaggle dataset mount point.
DATA_PATH = '/kaggle/input/datasets/siddhant22132/consumption-energy-east/PJME_hourly.csv'

# Fallback for local/testing runs
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'PJME_hourly.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
df.info()


## 2. Data Preprocessing

Steps:
- Parse `Datetime` to a proper timestamp and set as index
- Sort chronologically
- Check for duplicate timestamps
- Check for missing hours (gaps in the hourly frequency)
- Handle missing values


In [ ]:
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').reset_index(drop=True)
df = df.set_index('Datetime')

print(f"Date range: {df.index.min()}  ->  {df.index.max()}")
print(f"Total records: {len(df):,}")


In [ ]:
# Duplicate timestamps
n_dupes = df.index.duplicated().sum()
print(f"Duplicate timestamp rows: {n_dupes}")

if n_dupes > 0:
    df = df[~df.index.duplicated(keep='first')]
    print(f"Dropped duplicates. New shape: {df.shape}")


In [ ]:
# Check for missing hours by comparing to a complete hourly date range
full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
missing_hours = full_range.difference(df.index)
print(f"Expected hourly timestamps: {len(full_range):,}")
print(f"Missing timestamps: {len(missing_hours):,} ({len(missing_hours)/len(full_range)*100:.2f}%)")
missing_hours[:10]


In [ ]:
# Reindex to the full hourly range so gaps become explicit NaNs, then interpolate
df = df.reindex(full_range)
df.index.name = 'Datetime'

print(f"Missing values before interpolation: {df['PJME_MW'].isna().sum()}")

# Time-based interpolation is appropriate for a regularly-spaced load series
df['PJME_MW'] = df['PJME_MW'].interpolate(method='time')

print(f"Missing values after interpolation: {df['PJME_MW'].isna().sum()}")


In [ ]:
# Sanity check: look for implausible values (e.g. zero or negative load)
print(df['PJME_MW'].describe())
implausible = df[df['PJME_MW'] <= 0]
print(f"\nImplausible (<=0 MW) readings: {len(implausible)}")


## 3. Anomaly Detection

Simple statistical approach: flag points that fall far outside a rolling window's expected range
(rolling mean ± N rolling standard deviations). This is a first pass — more rigorous outlier
handling can be layered on later if needed.


In [ ]:
window = 24 * 7  # 1 week
roll_mean = df['PJME_MW'].rolling(window, center=True, min_periods=24).mean()
roll_std = df['PJME_MW'].rolling(window, center=True, min_periods=24).std()

z_threshold = 4
df['rolling_z'] = (df['PJME_MW'] - roll_mean) / roll_std
anomalies = df[df['rolling_z'].abs() > z_threshold]

print(f"Flagged anomalies: {len(anomalies)} ({len(anomalies)/len(df)*100:.3f}% of records)")
anomalies[['PJME_MW', 'rolling_z']].head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['PJME_MW'], linewidth=0.5, label='PJME Load (MW)')
ax.scatter(anomalies.index, anomalies['PJME_MW'], color='red', s=15, label='Flagged anomaly', zorder=5)
ax.set_title('PJM East Hourly Load with Flagged Anomalies')
ax.set_xlabel('Date')
ax.set_ylabel('Load (MW)')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['PJME_MW'], linewidth=0.4, color='steelblue')
ax.set_title('PJM East Hourly Electricity Load, 2002–2018')
ax.set_xlabel('Date')
ax.set_ylabel('Load (MW)')
plt.tight_layout()
plt.show()


In [ ]:
# Yearly overview
yearly = df['PJME_MW'].resample('Y').mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly.index.year, yearly.values, color='teal')
ax.set_title('Average Annual Load')
ax.set_xlabel('Year')
ax.set_ylabel('Average Load (MW)')
plt.tight_layout()
plt.show()


In [ ]:
# Monthly seasonality (across all years)
monthly_avg = df.groupby(df.index.month)['PJME_MW'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly_avg.index, monthly_avg.values, marker='o', color='darkorange')
ax.set_title('Average Load by Month (Seasonality)')
ax.set_xlabel('Month')
ax.set_ylabel('Average Load (MW)')
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()


In [ ]:
# Daily pattern: average load by hour of day
hourly_avg = df.groupby(df.index.hour)['PJME_MW'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly_avg.index, hourly_avg.values, marker='o', color='crimson')
ax.set_title('Average Load by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Average Load (MW)')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()


In [ ]:
# Weekly pattern: average load by day of week
dow_avg = df.groupby(df.index.dayofweek)['PJME_MW'].mean()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(dow_labels, dow_avg.values, color='mediumpurple')
ax.set_title('Average Load by Day of Week')
ax.set_ylabel('Average Load (MW)')
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap: hour of day vs. month, showing typical seasonal-daily interaction
pivot = df.pivot_table(values='PJME_MW', index=df.index.hour, columns=df.index.month, aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Avg Load (MW)'})
ax.set_title('Average Load by Hour of Day and Month')
ax.set_xlabel('Month')
ax.set_ylabel('Hour of Day')
plt.tight_layout()
plt.show()


## 5. Time-Series Decomposition

Using STL (Seasonal-Trend decomposition using LOESS) on a daily-resampled series to keep the
decomposition computationally manageable, and to make the trend/seasonal components easier to read.


In [ ]:
daily = df['PJME_MW'].resample('D').mean()

stl = STL(daily, period=365, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(daily.index, result.observed, color='steelblue'); axes[0].set_ylabel('Observed')
axes[1].plot(daily.index, result.trend, color='darkorange'); axes[1].set_ylabel('Trend')
axes[2].plot(daily.index, result.seasonal, color='seagreen'); axes[2].set_ylabel('Seasonal')
axes[3].plot(daily.index, result.resid, color='indianred'); axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')
fig.suptitle('STL Decomposition of Daily-Averaged PJM East Load')
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of the load values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['PJME_MW'], bins=80, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Hourly Load')
axes[0].set_xlabel('Load (MW)')

sns.boxplot(x=df.index.month, y=df['PJME_MW'], ax=axes[1], color='lightseagreen')
axes[1].set_title('Load Distribution by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Load (MW)')
plt.tight_layout()
plt.show()


## 6. Feature Engineering

### 6.1 Calendar Features

Hour, day-of-week, month, and a US federal holiday flag. These capture the daily and seasonal
demand cycles observed in the EDA above.


In [ ]:
feat = df.copy()

feat['hour'] = feat.index.hour
feat['dayofweek'] = feat.index.dayofweek
feat['day'] = feat.index.day
feat['month'] = feat.index.month
feat['quarter'] = feat.index.quarter
feat['year'] = feat.index.year
feat['dayofyear'] = feat.index.dayofyear
feat['weekofyear'] = feat.index.isocalendar().week.astype(int)
feat['is_weekend'] = (feat['dayofweek'] >= 5).astype(int)

# Cyclical encodings so models see hour 23 and hour 0 as adjacent
feat['hour_sin'] = np.sin(2 * np.pi * feat['hour'] / 24)
feat['hour_cos'] = np.cos(2 * np.pi * feat['hour'] / 24)
feat['month_sin'] = np.sin(2 * np.pi * feat['month'] / 12)
feat['month_cos'] = np.cos(2 * np.pi * feat['month'] / 12)

feat[['hour', 'dayofweek', 'month', 'is_weekend', 'hour_sin', 'hour_cos']].head()


In [ ]:
# US federal holiday flag
try:
    from pandas.tseries.holiday import USFederalHolidayCalendar
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=feat.index.min(), end=feat.index.max())
    feat['is_holiday'] = feat.index.normalize().isin(holidays).astype(int)
except ImportError:
    print("USFederalHolidayCalendar unavailable — install/enable pandas holiday calendar to add this feature.")
    feat['is_holiday'] = 0

print(f"Holiday hours flagged: {feat['is_holiday'].sum():,}")


### 6.2 Lag Features

24-hour (same hour yesterday) and 168-hour (same hour, same day, last week) lags, which are strong
naive predictors for load and useful inputs for the gradient boosting models.


In [ ]:
feat['lag_24h'] = feat['PJME_MW'].shift(24)
feat['lag_168h'] = feat['PJME_MW'].shift(168)

feat[['PJME_MW', 'lag_24h', 'lag_168h']].head(200).tail(10)


### 6.3 Rolling Window Statistics

7-day rolling mean and standard deviation to capture recent local trend and volatility.
Shifted by 1 hour to avoid leaking the current observation into its own rolling stats.


In [ ]:
roll_window = 24 * 7  # 7 days of hourly data

feat['rolling_mean_7d'] = feat['PJME_MW'].shift(1).rolling(roll_window).mean()
feat['rolling_std_7d'] = feat['PJME_MW'].shift(1).rolling(roll_window).std()

feat[['PJME_MW', 'rolling_mean_7d', 'rolling_std_7d']].tail(10)


In [ ]:
# Drop the warm-up rows that don't yet have full lag/rolling history
feat_clean = feat.dropna().copy()
print(f"Shape before dropping warm-up rows: {feat.shape}")
print(f"Shape after: {feat_clean.shape}")
feat_clean.head()


## 7. Feature Relationship Checks

Quick visual sanity check that the engineered lag and rolling features actually track the target,
before they're handed off to the modelling notebook.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

sample = feat_clean.loc['2010-01-01':'2010-02-01']
axes[0].plot(sample.index, sample['PJME_MW'], label='Actual', linewidth=1)
axes[0].plot(sample.index, sample['lag_168h'], label='Lag 168h', linewidth=1, alpha=0.7)
axes[0].set_title('Actual Load vs. 168h Lag (Jan 2010 sample)')
axes[0].legend()

axes[1].plot(sample.index, sample['PJME_MW'], label='Actual', linewidth=1)
axes[1].plot(sample.index, sample['rolling_mean_7d'], label='7-day Rolling Mean', linewidth=1, alpha=0.8)
axes[1].set_title('Actual Load vs. 7-day Rolling Mean (Jan 2010 sample)')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Correlation of engineered features with the target
corr_cols = ['PJME_MW', 'hour', 'dayofweek', 'month', 'is_weekend', 'is_holiday',
             'lag_24h', 'lag_168h', 'rolling_mean_7d', 'rolling_std_7d']
corr = feat_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix — Target vs. Engineered Features')
plt.tight_layout()
plt.show()


## 8. Save Feature Set

Persist the cleaned, feature-engineered dataframe for use in the modelling notebook (baselines,
XGBoost/LightGBM, Prophet, LSTM/TFT).


In [ ]:
OUTPUT_PATH = 'pjme_features.csv'
feat_clean.to_csv(OUTPUT_PATH)
print(f"Saved feature set with {feat_clean.shape[0]:,} rows and {feat_clean.shape[1]} columns to {OUTPUT_PATH}")
